### Imports

In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from tqdm import tqdm

### Data Preprocessing

In [2]:
DATA_PATH = "../../../data/processed/ratings_clean.csv"


df = pd.read_csv(DATA_PATH)
df = df[[
    'user_id',
    'item_id',
    'rating'
]]
print(df.head())

   user_id  item_id  rating
0        0    24418     5.0
1        1    52426     5.0
2        2   378332     3.0
3        2   368642     4.0
4        2   252413     5.0


In [3]:
num_users = df['user_id'].nunique()
num_items = df['item_id'].nunique()
num_ratings = len(df)

print(f"Users   : {num_users}")
print(f"Items   : {num_items}")
print(f"Ratings : {num_ratings}")

Users   : 6503429
Items   : 434043
Ratings : 17328314


In [4]:
MIN_USER_RATINGS = 10
MIN_ITEM_RATINGS = 10


# Filter users
user_counts = df['user_id'].value_counts()

active_users = user_counts[
    user_counts >= MIN_USER_RATINGS
].index

df = df[
    df['user_id'].isin(active_users)
]


# Filter items
item_counts = df['item_id'].value_counts()

popular_items = item_counts[
    item_counts >= MIN_ITEM_RATINGS
].index

df = df[
    df['item_id'].isin(popular_items)
]

In [5]:
num_users = df['user_id'].nunique()
num_items = df['item_id'].nunique()
num_ratings = len(df)

print(f"Users   : {num_users}")
print(f"Items   : {num_items}")
print(f"Ratings : {num_ratings}")

Users   : 243878
Items   : 100456
Ratings : 4719615


In [6]:
user_ids = df['user_id'].unique()
item_ids = df['item_id'].unique()


user_to_index = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}


item_to_index = {
    item_id: idx
    for idx, item_id in enumerate(item_ids)
}


index_to_user = {
    idx: user_id
    for user_id, idx in user_to_index.items()
}


index_to_item = {
    idx: item_id
    for item_id, idx in item_to_index.items()
}


# Create encoded columns

df['user_idx'] = df['user_id'].map(user_to_index)
df['item_idx'] = df['item_id'].map(item_to_index)


print(df.head())

    user_id  item_id  rating  user_idx  item_idx
29       10   237525     5.0         0         0
30       10    14644     5.0         0         1
31       10   337736     4.0         0         2
32       10   135026     4.0         0         3
33       10   205499     5.0         0         4


In [7]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

test_df, val_df = train_test_split(
    temp_df, 
    test_size=0.1,
    random_state=42
)

print(len(train_df))
print(len(test_df))
print(len(val_df))

3775692
849530
94393


In [8]:
class RatingsDataset(Dataset):

    def __init__(self, dataframe):

        self.users = torch.tensor(
            dataframe['user_idx'].values,
            dtype=torch.long
        )

        self.items = torch.tensor(
            dataframe['item_idx'].values,
            dtype=torch.long
        )

        self.ratings = torch.tensor(
            dataframe['rating'].values,
            dtype=torch.float32
        )

    def __len__(self):

        return len(self.ratings)

    def __getitem__(self, idx):

        return (
            self.users[idx],
            self.items[idx],
            self.ratings[idx]
        )

In [9]:
train_dataset = RatingsDataset(train_df)
test_dataset = RatingsDataset(test_df)
val_dataset = RatingsDataset(val_df)


train_loader = DataLoader(
    train_dataset,
    batch_size=2048,
    shuffle=True
)


test_loader = DataLoader(
    test_dataset,
    batch_size=2048,
    shuffle=False
)


val_loader = DataLoader(
    val_dataset,
    batch_size=2048,
    shuffle=False
)

### Collaborative Filtering Model

In [10]:
class CollaborativeFilteringModel(nn.Module):

    def __init__(
        self,
        num_users,
        num_items,
        embedding_dim=50
    ):

        super().__init__()

        # User embeddings
        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        # Item embeddings
        self.item_embedding = nn.Embedding(
            num_items,
            embedding_dim
        )

        # User bias
        self.user_bias = nn.Embedding(
            num_users,
            1
        )

        # Item bias
        self.item_bias = nn.Embedding(
            num_items,
            1
        )

    def forward(self, user_ids, item_ids):

        # Embedding lookup
        user_vecs = self.user_embedding(user_ids)
        item_vecs = self.item_embedding(item_ids)

        # Dot product
        interaction = (
            user_vecs * item_vecs
        ).sum(dim=1)

        # Biases
        user_b = self.user_bias(user_ids).squeeze()
        item_b = self.item_bias(item_ids).squeeze()

        prediction = (
            interaction
            + user_b
            + item_b
        )

        return prediction

In [11]:
num_users = len(user_to_index)
num_items = len(item_to_index)


model = CollaborativeFilteringModel(
    num_users=num_users,
    num_items=num_items,
    embedding_dim=50
)

print(model)

CollaborativeFilteringModel(
  (user_embedding): Embedding(243878, 50)
  (item_embedding): Embedding(100456, 50)
  (user_bias): Embedding(243878, 1)
  (item_bias): Embedding(100456, 1)
)


### Training Setup

In [12]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(DEVICE)

model = model.to(DEVICE)

cuda


In [18]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

### Training

In [ ]:
EPOCHS = 50

PATIENCE = 3

best_val_loss = float('inf')

patience_counter = 0


for epoch in range(EPOCHS):

    # =========================
    # TRAINING
    # =========================

    model.train()

    train_loss = 0

    for users, items, ratings in tqdm(
        train_loader,
        desc=f"Training Epoch {epoch+1}"
    ):

        users = users.to(DEVICE)
        items = items.to(DEVICE)
        ratings = ratings.to(DEVICE)

        predictions = model(
            users,
            items
        )

        loss = criterion(
            predictions,
            ratings
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = (
        train_loss / len(train_loader)
    )


    # =========================
    # VALIDATION
    # =========================

    model.eval()

    val_loss = 0

    with torch.no_grad():

        for users, items, ratings in tqdm(
            val_loader,
            desc=f"Validation Epoch {epoch+1}"
        ):

            users = users.to(DEVICE)
            items = items.to(DEVICE)
            ratings = ratings.to(DEVICE)

            predictions = model(
                users,
                items
            )

            loss = criterion(
                predictions,
                ratings
            )

            val_loss += loss.item()

    avg_val_loss = (
        val_loss / len(val_loader)
    )


    # =========================
    # LOGGING
    # =========================

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}"
    )


    # =========================
    # EARLY STOPPING
    # =========================

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        patience_counter = 0

        # Save best model
        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

        print("Validation improved → model saved")

    else:

        patience_counter += 1

        print(
            f"No improvement "
            f"({patience_counter}/{PATIENCE})"
        )

        if patience_counter >= PATIENCE:

            print("Early stopping triggered")

            break

Validation Epoch 1: 100%|██████████████████████████████████████████████████████████████| 47/47 [00:03<00:00, 14.74it/s]


Epoch 1 | Train Loss: 54.3071 | Val Loss: 47.7757
Validation improved → model saved


Validation Epoch 2: 100%|██████████████████████████████████████████████████████████████| 47/47 [00:03<00:00, 14.33it/s]


Epoch 2 | Train Loss: 38.3356 | Val Loss: 38.6378
Validation improved → model saved


Validation Epoch 3: 100%|██████████████████████████████████████████████████████████████| 47/47 [00:03<00:00, 12.47it/s]


...

Validation Epoch 34: 100%|█████████████████████████████████████████████████████████████| 47/47 [00:03<00:00, 11.76it/s]


Epoch 34 | Train Loss: 1.9454 | Val Loss: 2.1958
Validation improved → model saved


Validation Epoch 35: 100%|█████████████████████████████████████████████████████████████| 47/47 [00:02<00:00, 16.16it/s]


Epoch 35 | Train Loss: 1.9427 | Val Loss: 2.1943
Validation improved → model saved


Training Epoch 36:   4%|██▍                                                          | 72/1844 [00:20<08:12,  3.60it/s]

In [19]:
EPOCHS = 30


for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for users, items, ratings in tqdm(train_loader):

        users = users.to(DEVICE)
        items = items.to(DEVICE)
        ratings = ratings.to(DEVICE)

        predictions = model(
            users,
            items
        )

        loss = criterion(
            predictions,
            ratings
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1} | Loss: {avg_loss:.4f}"
    )

100%|██████████████████████████████████████████████████████████████████████████████| 1844/1844 [01:21<00:00, 22.49it/s]


Epoch 1 | Loss: 2.7448


100%|██████████████████████████████████████████████████████████████████████████████| 1844/1844 [01:18<00:00, 23.34it/s]


Epoch 2 | Loss: 2.3221


100%|██████████████████████████████████████████████████████████████████████████████| 1844/1844 [01:17<00:00, 23.90it/s]


...

Epoch 28 | Loss: 1.9130


100%|██████████████████████████████████████████████████████████████████████████████| 1844/1844 [01:19<00:00, 23.29it/s]


Epoch 29 | Loss: 1.9131


100%|██████████████████████████████████████████████████████████████████████████████| 1844/1844 [01:19<00:00, 23.28it/s]

Epoch 30 | Loss: 1.9119


In [14]:
model.load_state_dict(
    torch.load(
        "best_model.pth",
        map_location=DEVICE
    )
)

model.eval()

print("Best model loaded.")

Best model loaded.


C:\Users\Aman\AppData\Local\Temp\ipykernel_5072\3148119541.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


In [15]:
from sklearn.metrics import mean_squared_error

model.eval()

all_predictions = []
all_targets = []

test_loss = 0


with torch.no_grad():

    for users, items, ratings in tqdm(test_loader):

        users = users.to(
            DEVICE,
            non_blocking=True
        )

        items = items.to(
            DEVICE,
            non_blocking=True
        )

        ratings = ratings.to(
            DEVICE,
            non_blocking=True
        )

        predictions = model(
            users,
            items
        )

        loss = criterion(
            predictions,
            ratings
        )

        test_loss += loss.item()

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_targets.extend(
            ratings.cpu().numpy()
        )


avg_test_loss = (
    test_loss / len(test_loader)
)

rmse = np.sqrt(
    mean_squared_error(
        all_targets,
        all_predictions
    )
)

print(f"Test Loss : {avg_test_loss:.4f}")
print(f"Test RMSE : {rmse:.4f}")

100%|████████████████████████████████████████████████████████████████████████████████| 415/415 [00:11<00:00, 35.18it/s]


Test Loss : 2.2006
Test RMSE : 1.4834


In [ ]:
model.eval()

all_predictions = []
all_targets = []

with torch.no_grad():

    for users, items, ratings in test_loader:

        users = users.to(DEVICE)
        items = items.to(DEVICE)

        predictions = model(
            users,
            items
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_targets.extend(
            ratings.numpy()
        )


rmse = np.sqrt(
    mean_squared_error(
        all_targets,
        all_predictions
    )
)

print(f"RMSE: {rmse:.4f}")

In [16]:
def recommend_movies(
    user_id,
    top_k=10
):

    model.eval()

    user_idx = user_to_index[user_id]

    watched_movies = set(
        df[
            df['user_id'] == user_id
        ]['item_id']
    )

    candidate_movies = [
        item_id
        for item_id in item_to_index.keys()
        if item_id not in watched_movies
    ]

    predictions = []

    with torch.no_grad():

        for item_id in candidate_movies:

            item_idx = item_to_index[item_id]

            user_tensor = torch.tensor(
                [user_idx],
                dtype=torch.long
            ).to(DEVICE)

            item_tensor = torch.tensor(
                [item_idx],
                dtype=torch.long
            ).to(DEVICE)

            prediction = model(
                user_tensor,
                item_tensor
            )

            predictions.append(
                (
                    item_id,
                    prediction.item()
                )
            )

    predictions.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return predictions[:top_k]

In [17]:
sample_user = df['user_id'].iloc[0]

recommendations = recommend_movies(
    sample_user,
    top_k=10
)

recommendations

[(np.int64(311284), 5.020259857177734),
 (np.int64(13637), 4.94162654876709),
 (np.int64(147210), 4.927561283111572),
 (np.int64(289486), 4.897871017456055),
 (np.int64(284668), 4.896798133850098),
 (np.int64(272137), 4.890900611877441),
 (np.int64(295674), 4.890742301940918),
 (np.int64(311148), 4.883054733276367),
 (np.int64(274467), 4.873553276062012),
 (np.int64(334935), 4.86344051361084)]